In [1]:
import re

import pandas as pd

In [2]:
file_name = '../../data/2018-2025/ncsl_ai_db_2018_2025_clean.csv'
df = pd.read_csv(file_name)

# Fill in empty strings for null text columns
df["title"] = df.title.fillna("")
df["summary"] = df.summary.fillna("")

In [3]:
# Map the bill status sub-types to main types - {Passed, Failed, Pending, Vetoed}
bill_status_map = {
    "Enacted": "Passed",
    "Failed": "Failed",
    "Failed - Adjourned": "Failed",
    "Adopted": "Passed",
    "Pending": "Pending",
    "Pending - Carryover": "Pending",
    "To Governor": "Pending",
    "Override Pending": "Pending",
    "Vetoed": "Vetoed"
}
df["status_type"] = df.status.map(bill_status_map)

unmapped_status = df.loc[df.status_type.isna(), "status"].unique()
assert len(
    unmapped_status) == 0, f"Statuses missing from bill_status_map: {unmapped_status}"

# Bill outcome flags
df["outcome_final"] = df.status_type != "Pending"
df["status_stale"] = (df.status_type == "Pending") & (
    df.page_year < df.page_year.max())

In [4]:
# Map the audience from the bill_number to the session's chamber
bill_audience_map = {'A': 'Assembly Bill',
                     'ACR': 'Assembly Concurrent Resolution',
                     'AJR': 'Assembly Joint Resolution',
                     'AR': 'Assembly Resolution',
                     'B': 'Bill',
                     'BDR': 'Bill Draft Request',
                     'D': 'Draft Bill',
                     'H': 'House Bill',
                     'HCR': 'House Concurrent Resolution',
                     'HD': 'House Docket',
                     'HJM': 'House Joint Memorial',
                     'HJR': 'House Joint Resolution',
                     'HM': 'House Memorial',
                     'HR': 'House Resolution',
                     'HSB': 'House Study Bill',
                     'L': 'Legislative Bill',
                     'LCR': 'Legislative Council Resolution',
                     'LR': 'Legislative Reference',
                     'S': 'Senate Bill',
                     'SCR': 'Senate Concurrent Resolution',
                     'SD': 'Senate Docket',
                     'SJR': 'Senate Joint Resolution',
                     'SM': 'Senate Memorial',
                     'SPB': 'Senate Bill',
                     'SR': 'Senate Resolution',
                     'SSB': 'Senate Study Bill',
                     'WS': 'Senate Bill'}

# Split 'bill_number' into chamber prefix and bill number into columns
prefix_number = df.bill_number.str.extract(r"^([A-Za-z]+)\s*(\d+)")
unparsed = df.loc[prefix_number.isna().any(axis=1), "bill_number"].unique()
assert len(
    unparsed) == 0, f"Bill numbers without a prefix and number: {unparsed}"


# Get full chamber name from the prefix
df["chamber_prefix"] = prefix_number[0].str.upper()
df["chamber_name"] = df.chamber_prefix.map(bill_audience_map)
unmapped_prefix = df.loc[df.chamber_name.isna(), "chamber_prefix"].unique()
assert len(
    unmapped_prefix) == 0, f"Prefixes missing from bill_audience_map: {unmapped_prefix}"

# Clean bill number and create an integer type column for order bill introduced
df["order_introduced"] = prefix_number[1].astype(int)
df["bill_number"] = df.bill_number.str.replace(" ", "")

In [5]:
# Chamber (from the bill prefix). "Other" = drafts, legislative references, DC/territory bills
df["chamber"] = df.chamber_prefix.str[0].map(
    {"H": "House", "S": "Senate", "A": "Assembly"}).fillna("Other")
df.loc[df.chamber_prefix == "WS", "chamber"] = "Senate"
df["is_lower_chamber"] = df.chamber.isin(["House", "Assembly"])

# Correct resolutions typed as bills (RI/SC use bill-style prefixes; their summary starts with a "(Resolution)" tag)
res_tag = r"^\((?:Senate |House )?Resolution\)\s*"
is_tagged = df.summary.str.contains(res_tag, regex=True)
df.loc[is_tagged, "chamber_name"] = df.loc[is_tagged,
                                           "chamber_name"].str.replace("Bill", "Resolution")
df["summary"] = df.summary.str.replace(res_tag, "", regex=True)

In [6]:
document_types = ["Resolution", "Study Bill",
                  "Memorial", "Docket", "Draft", "Reference"]

pattern = f"(?i)({'|'.join(document_types)})"

# Extract matching document type, with "Bill" as the default (now that bill-types are fixed)
df["document_type"] = df.chamber_name.str.extract(
    pattern, expand=False).fillna("Bill")

In [7]:
# Encode different types of legislation
df["is_resolution"] = df.chamber_name.str.contains("Resolution")
df["is_joint_resolution"] = df.chamber_name.str.contains("Joint Resolution")
df["is_concurrent_resolution"] = df.chamber_name.str.contains(
    "Concurrent Resolution")
df["is_simple_resolution"] = df.chamber_prefix.isin(["HR", "SR", "AR"])

In [8]:
# Gets the type of each state/area - {State, Territory, District}
df['jurisdiction_type'] = df.state.map(
    lambda s: "Territory" if s in {"PR", "VI", "GU", "AS", "MP"}
    else "District" if s == "DC" else "State")

region_map = {
    **dict.fromkeys(["CT", "ME", "MA", "NH", "RI", "VT", "NJ", "NY", "PA"], "Northeast"),
    **dict.fromkeys(["IL", "IN", "MI", "OH", "WI", "IA", "KS", "MN", "MO", "NE", "ND", "SD"], "Midwest"),
    **dict.fromkeys(["DE", "DC", "FL", "GA", "MD", "NC", "SC", "VA", "WV", "AL", "KY", "MS", "TN",
                     "AR", "LA", "OK", "TX"], "South"),
    **dict.fromkeys(["AZ", "CO", "ID", "MT", "NV", "NM", "UT", "WY", "AK", "CA", "HI", "OR", "WA"], "West"),
    **dict.fromkeys(["PR", "VI", "GU", "AS", "MP"], "Territory"),
}

df["census_region"] = df.state.map(region_map)
assert df.census_region.notna().all(
), df.loc[df.census_region.isna(), "state"].unique()

In [9]:
def any_of(names):
    """1 if the bill has any of these categories (a category no bill has counts as 0)"""
    cols = [f"category_{n}" for n in names]
    return df.reindex(columns=cols, fill_value=0).max(axis=1)


# One hot encoding categories from multi-label text
# Split strings by separator, strip whitespace, and create dummy variables
df.category = df.category.fillna("").str.lower()
dummies = df.category.str.get_dummies(sep=";")
dummies = dummies.rename(columns=lambda c: "category_" +
                         re.sub(r"\W+", "_", c).strip("_"))
# blank label from uncategorized bills, not a category
dummies = dummies.drop(columns="category_", errors="ignore")

In [10]:
# Total number of categories
df["num_categories"] = dummies.sum(axis=1)
df["is_uncategorized"] = df.num_categories == 0

# Add new category columns to existing dataset
df = pd.concat([df, dummies], axis=1)

# Whether a bill was for study-use only
df["studies_only"] = (any_of(["studies"]) == 1) & (df.num_categories == 1)

In [11]:
# Category groups (from the one-hot flags above) and a flag for bills NCSL left uncategorized.
df["catgroup_public_sector"] = any_of(
    ["government_use", "judicial_use", "appropriations", "audit"])
df["catgroup_regulatory"] = any_of(["private_sector", "notification", "private_right_of_action",
                                    "impact_assessment", "oversight_governance", "responsible_use"])
df["catgroup_rights_safety"] = any_of(
    ["criminal_use", "child_pornography", "elections", "personhood"])
df["catgroup_sector_specific"] = any_of(["health_use", "education_use", "education_training", "housing",
                                         "energy", "effect_on_labor_employment", "cybersecurity", "taxes"])

# 'Education Use' only exists on the 2024+ pages, so catgroup_education works across years
df["catgroup_education"] = any_of(["education_use", "education_training"])

In [12]:
# Text structure features
text = (df.title + ". " + df.summary.fillna("")).str.strip()

keyword_flags = {
    "deepfake_synthetic": r"deep ?fakes?|synthetic media|digital replicas?|digital forger|\blikeness",
    "facial_recognition_biometric": r"facial recognition|\bbiometric",
    "chatbot_companion": r"\bchat ?bots?\b|\bcompanion|\bconversational",
    "generative_ai": r"\bgenerative",
    "autonomous_vehicle": r"autonomous vehicles?|self-driving|driverless",
    "algorithmic_discrimination": r"algorithmic discrimination|\bdiscriminat|\bbias(?:es|ed)?\b",
    "employment": r"\bemploy|\bhiring\b|\bworkers?\b|\blabor\b",
    "health_care": r"\bhealth|\bmedical|\bpatients?\b|\bphysicians?\b|\binsur",
    "education": r"\bschools?\b|\bstudents?\b|\beducat|\buniversit",
    "elections_political": r"\belections?\b|\bcampaigns?\b|\bpolitical\b|\bcandidates?\b",
    "child_minor": r"\bchild|\bminors?\b|\bjuvenile",
    "government_use": r"state agenc|\bgovernment|public sector|\bprocure|\binventor(?:y|ies)\b",
    "task_force_study": r"task force|\bstud(?:y|ies)\b|\bcommission\b|working group|advisory (?:council|committee)",
    "appropriation": r"\bappropriat|\bfunds?\b|\bfunding\b|\bgrants?\b|\$",
    "disclosure_transparency": r"\bdisclos|\btransparen|\bwatermark|\bnotice\b|\bnotif",
    "penalty_criminal": r"\bpenalt|\bfines?\b|\bfelony|\bmisdemeanor|\bcrimes?\b|\bcriminal|\boffenses?\b",
    "private_right_of_action": r"private right of action|civil action|\bliab",
    "definition": r"\bdefin",  # mentions a definition (not "definition only")
    "privacy_data": r"\bprivacy|personal data|personal information|data protection",
}
for name, pattern in keyword_flags.items():
    df[f"mentions_{name}"] = text.str.contains(pattern, case=False, regex=True)

# Check % of bills each menion_ flag hits
# print(df.filter(like="mentions_").mean().sort_values(ascending=False).round(3))

In [13]:
# Final data quality checks
assert df.bill_id.is_unique
assert (df.session_year <= df.page_year).all()
assert df.state.notna().all() and df.bill_number.notna().all()

df = df.drop(columns=["category", "chamber_name", "page_year"])

missing = df.isnull().sum()[lambda s: s > 0]
assert missing.empty, f"Missing values:\n{missing}"

# Export final csv to dataset
file_name = "../../data/2018-2025/ncsl_ai_db_2018_2025_final.csv"
df.to_csv(file_name, index=False)

# View final columns for 2018 - 2025 bills dataset
print(df.columns)

Index(['bill_id', 'bill_number', 'state', 'session_year', 'status', 'title',
       'summary', 'is_special_session', 'status_type', 'outcome_final',
       'status_stale', 'chamber_prefix', 'order_introduced', 'chamber',
       'is_lower_chamber', 'document_type', 'is_resolution',
       'is_joint_resolution', 'is_concurrent_resolution',
       'is_simple_resolution', 'jurisdiction_type', 'census_region',
       'num_categories', 'is_uncategorized', 'category_appropriations',
       'category_audit', 'category_child_pornography', 'category_criminal_use',
       'category_cybersecurity', 'category_education_use',
       'category_education_training', 'category_effect_on_labor_employment',
       'category_elections', 'category_energy', 'category_government_use',
       'category_health_use', 'category_housing', 'category_impact_assessment',
       'category_judicial_use', 'category_notification',
       'category_oversight_governance', 'category_personhood',
       'category_private_rig